# JS09 Tugas Lab 1: Perbandingan Performansi ANNOY, FAISS, dan HNSW

Tugas ini mengulang benchmark pada empat kombinasi ukuran dan dimensi data, masing-masing dengan dua metric jarak:

| Jumlah Data/Dimensi | Metric |
| ------------------- | ------ |
| 1000 / 2D           | Euclidean, Cosine |
| 1000 / 5D           | Euclidean, Cosine |
| 1000000 / 2D        | Euclidean, Cosine |
| 1000000 / 5D        | Euclidean, Cosine |

Setiap kombinasi diuji pada ANNOY, FAISS, dan HNSW. Waktu yang dicatat adalah waktu membangun indeks dan waktu satu query untuk `k=10`.

Catatan metric:

- ANNOY memakai `euclidean` dan `angular`. `angular` adalah padanan cosine untuk Annoy.
- FAISS memakai `IndexFlatL2` untuk Euclidean dan `IndexFlatIP` pada vektor yang sudah dinormalisasi untuk cosine.
- HNSW memakai `space='l2'` dan `space='cosine'`.

## 1. Import Library dan Definisi Fungsi

In [1]:
import time
import numpy as np
import pandas as pd
import faiss
from annoy import AnnoyIndex
import hnswlib

K = 10
TREES = 10
EF_CONSTRUCTION = 100
M = 16
EF_SEARCH = 50
REPEATS = 20  # satu query terlalu cepat untuk diukur sekali


def normalize(X):
    return X / np.linalg.norm(X, axis=1, keepdims=True)


def time_query(fn, repeats=REPEATS):
    """Jalankan query berulang, kembalikan hasil terakhir dan rata-rata waktunya."""
    start = time.time()
    for _ in range(repeats):
        out = fn()
    return out, (time.time() - start) / repeats


def run_annoy(X, query, metric):
    space = "euclidean" if metric == "euclidean" else "angular"
    index = AnnoyIndex(X.shape[1], space)
    for i, v in enumerate(X):
        index.add_item(i, v)

    start = time.time()
    index.build(TREES)
    build_time = time.time() - start

    idx, query_time = time_query(lambda: index.get_nns_by_vector(query, K, search_k=1000))
    return idx, build_time, query_time


def run_faiss(X, query, metric):
    if metric == "euclidean":
        index = faiss.IndexFlatL2(X.shape[1])
        data, q = X, query.reshape(1, -1)
    else:
        index = faiss.IndexFlatIP(X.shape[1])
        data, q = normalize(X), normalize(query.reshape(1, -1))

    start = time.time()
    index.add(data)
    build_time = time.time() - start

    (_, idx), query_time = time_query(lambda: index.search(q, K))
    return idx[0].tolist(), build_time, query_time


def run_hnsw(X, query, metric):
    space = "l2" if metric == "euclidean" else "cosine"
    index = hnswlib.Index(space=space, dim=X.shape[1])
    index.init_index(max_elements=len(X), ef_construction=EF_CONSTRUCTION, M=M)

    start = time.time()
    index.add_items(X)
    build_time = time.time() - start

    index.set_ef(EF_SEARCH)
    (idx, _), query_time = time_query(lambda: index.knn_query(query, k=K))
    return idx[0].tolist(), build_time, query_time

## 2. Jalankan Seluruh Kombinasi

Data dibangkitkan ulang untuk setiap kombinasi supaya tidak ada indeks lama yang tertinggal di memori. Query diambil dari titik acak yang sama untuk ketiga algoritma.

In [2]:
configs = [(1_000, 2), (1_000, 5), (1_000_000, 2), (1_000_000, 5)]
metrics = ["euclidean", "cosine"]

rows = []
rng = np.random.default_rng(42)

for n_data, dim in configs:
    X = rng.random((n_data, dim), dtype=np.float32)
    query = rng.random(dim, dtype=np.float32)

    for metric in metrics:
        for method, runner in [("ANNOY", run_annoy), ("FAISS", run_faiss), ("HNSW", run_hnsw)]:
            idx, build_time, query_time = runner(X, query, metric)
            rows.append({
                "n_data": n_data,
                "dim": dim,
                "metric": metric,
                "method": method,
                "build_s": round(build_time, 4),
                "query_ms": round(query_time * 1000, 4),
                "top5": idx[:5],
            })
            print(f"{n_data:>7} data | {dim}D | {metric:<9} | {method:<5} | "
                  f"build {build_time:8.4f} s | query {query_time * 1000:8.4f} ms")

results = pd.DataFrame(rows)
results

   1000 data | 2D | euclidean | ANNOY | build   0.0067 s | query   0.0500 ms
   1000 data | 2D | euclidean | FAISS | build   0.0000 s | query   0.0500 ms
   1000 data | 2D | euclidean | HNSW  | build   0.0065 s | query   0.0000 ms
   1000 data | 2D | cosine    | ANNOY | build   0.0072 s | query   0.0500 ms
   1000 data | 2D | cosine    | FAISS | build   0.0000 s | query   0.0000 ms
   1000 data | 2D | cosine    | HNSW  | build   0.0050 s | query   0.0000 ms
   1000 data | 5D | euclidean | ANNOY | build   0.0065 s | query   0.0502 ms
   1000 data | 5D | euclidean | FAISS | build   0.0000 s | query   0.0000 ms
   1000 data | 5D | euclidean | HNSW  | build   0.0060 s | query   0.0500 ms
   1000 data | 5D | cosine    | ANNOY | build   0.0053 s | query   0.0578 ms
   1000 data | 5D | cosine    | FAISS | build   0.0000 s | query   0.0000 ms
   1000 data | 5D | cosine    | HNSW  | build   0.0060 s | query   0.0000 ms


1000000 data | 2D | euclidean | ANNOY | build  10.0480 s | query   0.2638 ms
1000000 data | 2D | euclidean | FAISS | build   0.0015 s | query   4.2721 ms


1000000 data | 2D | euclidean | HNSW  | build  24.3047 s | query   0.0500 ms


1000000 data | 2D | cosine    | ANNOY | build  41.7815 s | query   0.1968 ms
1000000 data | 2D | cosine    | FAISS | build   0.0030 s | query   2.2573 ms


1000000 data | 2D | cosine    | HNSW  | build  41.2447 s | query   0.0497 ms


1000000 data | 5D | euclidean | ANNOY | build   9.8238 s | query   0.2501 ms
1000000 data | 5D | euclidean | FAISS | build   0.0189 s | query   8.2792 ms


1000000 data | 5D | euclidean | HNSW  | build  37.2086 s | query   0.0514 ms


1000000 data | 5D | cosine    | ANNOY | build  13.8671 s | query   0.2260 ms
1000000 data | 5D | cosine    | FAISS | build   0.0095 s | query   3.1808 ms


1000000 data | 5D | cosine    | HNSW  | build  33.4203 s | query   0.0501 ms


,n_data,dim,metric,method,build_s,query_ms,top5
0,1000,2,euclidean,ANNOY,0.0067,0.0500,"[258, 17, 971, 187, 794]"
1,1000,2,euclidean,FAISS,0.0000,0.0500,"[258, 17, 971, 187, 794]"
2,1000,2,euclidean,HNSW,0.0065,0.0000,"[258, 17, 971, 187, 794]"
3,1000,2,cosine,ANNOY,0.0072,0.0500,"[74, 820, 17, 84, 512]"
4,1000,2,cosine,FAISS,0.0000,0.0000,"[74, 820, 17, 84, 512]"
5,1000,2,cosine,HNSW,0.0050,0.0000,"[74, 820, 17, 84, 512]"
6,1000,5,euclidean,ANNOY,0.0065,0.0502,"[421, 98, 734, 13, 825]"
7,1000,5,euclidean,FAISS,0.0000,0.0000,"[421, 98, 734, 13, 825]"
8,1000,5,euclidean,HNSW,0.0060,0.0500,"[421, 98, 734, 13, 825]"
9,1000,5,cosine,ANNOY,0.0053,0.0578,"[421, 362, 825, 98, 598]"


## 3. Tabel Performansi

Tabel pertama merangkum waktu query, tabel kedua merangkum waktu build. Satuan waktu query adalah milidetik.

In [3]:
query_table = results.pivot_table(
    index=["n_data", "dim", "metric"], columns="method", values="query_ms"
).round(4)
query_table

method                  ANNOY   FAISS    HNSW
n_data  dim metric                           
1000    2   cosine     0.0500  0.0000  0.0000
            euclidean  0.0500  0.0500  0.0000
        5   cosine     0.0578  0.0000  0.0000
            euclidean  0.0502  0.0000  0.0500
1000000 2   cosine     0.1968  2.2573  0.0497
            euclidean  0.2638  4.2721  0.0500
        5   cosine     0.2260  3.1808  0.0501
            euclidean  0.2501  8.2792  0.0514

In [4]:
build_table = results.pivot_table(
    index=["n_data", "dim", "metric"], columns="method", values="build_s"
).round(4)
build_table

method                   ANNOY   FAISS     HNSW
n_data  dim metric                             
1000    2   cosine      0.0072  0.0000   0.0050
            euclidean   0.0067  0.0000   0.0065
        5   cosine      0.0053  0.0000   0.0060
            euclidean   0.0065  0.0000   0.0060
1000000 2   cosine     41.7815  0.0030  41.2447
            euclidean  10.0480  0.0015  24.3047
        5   cosine     13.8671  0.0095  33.4203
            euclidean   9.8238  0.0189  37.2086

Dari tabel terlihat bahwa FAISS Flat hampir tidak butuh waktu build, tetapi waktu query-nya naik seiring jumlah data karena tetap memeriksa semua vektor. ANNOY dan HNSW butuh waktu build lebih lama, namun waktu query-nya jauh lebih kecil dan relatif stabil ketika data bertambah.